# Preparation des donnees -- JO 2028 (version simplifiee)

---

## Objectif

Construire **un seul dataset propre et agrege** pour la prediction :
`medals_per_country_sport.csv` -- 1 ligne = (NOC, Year, Sport, Gold, Silver, Bronze).

## Pourquoi cette simplification ?

Le brief demande de predire :
1. Le **pays gagnant** en 2028
2. Les **medailles par sport et type** (Or/Argent/Bronze)

On n'a donc PAS besoin :
- Des informations individuelles athletes (Age, Height, Weight)
- Des fichiers `bios.csv`, `bios_locs.csv` (~25 Mo)
- Des analyses par genre, age, mensurations
- Des notebooks de stats descriptives biometriques

On garde uniquement les fichiers necessaires :
- `results.csv` -- 308 408 participations avec medailles
- `olympics_1896_2024.csv` -- pour Paris 2024 (KeithGalli s'arrete a 2022)
- `noc_regions.csv` -- mapping NOC -> Country
- `populations.csv` -- pour la feature population


In [1]:
# === Imports ===
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Configuration d'affichage
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

# Chemins
RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. Chargement des sources

On charge uniquement les 4 fichiers necessaires pour la prediction par (pays, sport).
**Aucun fichier biographique** (`bios.csv`, `bios_locs.csv`) n'est utilise.

In [2]:
# Chargement des 4 sources (sans les bios)
results = pd.read_csv(os.path.join(RAW_DIR, "results.csv"), low_memory=False)
olympics = pd.read_csv(os.path.join(RAW_DIR, "olympics_1896_2024.csv"))
noc_regions = pd.read_csv(os.path.join(RAW_DIR, "noc_regions.csv"))
populations = pd.read_csv(os.path.join(RAW_DIR, "populations.csv"))

# Normalisation des noms de colonnes results (KeithGalli minuscules)
results = results.rename(columns={
    "year": "Year", "type": "Season", "discipline": "Sport",
    "event": "Event", "noc": "NOC", "medal": "Medal",
})

print(f"results               : {results.shape[0]:>8,} lignes")
print(f"olympics_1896_2024    : {olympics.shape[0]:>8,} lignes")
print(f"noc_regions           : {noc_regions.shape[0]:>8,} lignes")
print(f"populations           : {populations.shape[0]:>8,} lignes")

results               :  308,408 lignes
olympics_1896_2024    :   24,012 lignes
noc_regions           :      230 lignes
populations           :      266 lignes


## 2. Nettoyage de `results.csv`

On filtre uniquement les **JO d'ete medailles** (Paris 2028 est un JO d'ete a LA).
On supprime les doublons exacts.

In [3]:
# Filtre : JO d'ete uniquement, annees olympiques valides
TRUE_SUMMER_YEARS = {1896,1900,1904,1908,1912,1920,1924,1928,1932,1936,1948,
                     1952,1956,1960,1964,1968,1972,1976,1980,1984,1988,1992,
                     1996,2000,2004,2008,2012,2016,2020,2024}

results = results[(results["Season"] == "Summer") &
                   (results["Year"].isin(TRUE_SUMMER_YEARS))].copy()

# Suppression des doublons exacts (bugs de scraping connus : Sailing 1900, Golf 1904)
n_before = len(results)
results = results.drop_duplicates().reset_index(drop=True)
print(f"Doublons exacts supprimes : {n_before - len(results):,}")

# On garde uniquement les medailles (les non-medailles ne servent pas pour notre cible)
medals_only = results[results["Medal"].notna()].copy()
print(f"\nLignes medailles : {len(medals_only):,}")
print(f"Annees couvertes : {medals_only['Year'].min()} - {medals_only['Year'].max()}")
print(f"\nRepartition par type :")
print(medals_only['Medal'].value_counts().to_string())

Doublons exacts supprimes : 124

Lignes medailles : 35,522
Annees couvertes : 1896.0 - 2020.0

Repartition par type :
Medal
Bronze    11975
Gold      11869
Silver    11678


## 3. Integration de Paris 2024

`results.csv` (KeithGalli) s'arrete en **2022**. On ajoute les medailles 2024
depuis `olympics_1896_2024.csv` qui couvre toutes les editions jusqu'a Paris.

In [4]:
# Filtrer 2024 dans olympics
paris_2024 = olympics[olympics["event_year"] == 2024].copy()

# Deduplication : 1 medaille = 1 (discipline, event, medal_type, country_code).
# Sans cela, chaque athlete d'equipe compte (1 medaille de relais 4x100 = 4 lignes).
paris_unique = paris_2024.drop_duplicates(
    subset=["discipline", "event", "medal_type", "country_code"]
)

# Mapper au format de results : Year, Sport, NOC, Medal
paris_mapped = pd.DataFrame({
    "Year": 2024,
    "Sport": paris_unique["discipline"],
    "NOC": paris_unique["country_code"],
    "Medal": paris_unique["medal_type"],
})

# Pour coherence : faire le meme traitement sur les autres annees de results
# (un relais d'or = 1 medaille pour le pays, pas N lignes)
medals_clean = medals_only[medals_only["Year"] != 2024][["Year", "Sport", "Event", "NOC", "Medal"]].copy()
medals_clean = medals_clean.drop_duplicates(subset=["Year", "Sport", "Event", "NOC", "Medal"])
medals_clean = medals_clean[["Year", "Sport", "NOC", "Medal"]]

# Concatener
medals_clean = pd.concat([medals_clean, paris_mapped], ignore_index=True)

print(f"Medailles 2024 dedupliquees : {len(paris_mapped):,} (etait {len(paris_2024)} avec les athletes equipe)")
print(f"\nDataset total : {len(medals_clean):,} lignes")
print(f"\nAnnees uniques : {medals_clean['Year'].nunique()} editions")


Medailles 2024 dedupliquees : 1,043 (etait 2315 avec les athletes equipe)

Dataset total : 17,802 lignes

Annees uniques : 30 editions


## 4. Agregation par (NOC, Year, Sport)

On transforme les lignes individuelles en **comptages** :
chaque ligne du dataset final represente le nombre de medailles Or/Argent/Bronze
qu'un pays a remporte dans un sport donne lors d'une edition donnee.

In [5]:
# Compter par (NOC, Year, Sport, Medal) puis pivoter
agg = medals_clean.groupby(["NOC", "Year", "Sport", "Medal"]).size().unstack(fill_value=0).reset_index()

# Assurer les 3 colonnes Gold/Silver/Bronze
for col in ["Gold", "Silver", "Bronze"]:
    if col not in agg.columns:
        agg[col] = 0

# Total = Or + Argent + Bronze
agg["Total"] = agg["Gold"] + agg["Silver"] + agg["Bronze"]

# Ajouter le nom du pays via noc_regions
agg = agg.merge(noc_regions[["NOC", "region"]].rename(columns={"region": "Country"}),
                  on="NOC", how="left")

# Reorganiser les colonnes
agg = agg[["Year", "NOC", "Country", "Sport", "Gold", "Silver", "Bronze", "Total"]]
agg = agg.sort_values(["Year", "NOC", "Sport"]).reset_index(drop=True)

print(f"Dataset agrege : {agg.shape}")
print(f"\nApercu :")
print(agg.head(10).to_string(index=False))

Dataset agrege : (6871, 8)

Apercu :
   Year NOC   Country                  Sport  Gold  Silver  Bronze  Total
1896.00 AUS Australia              Athletics     2       0       1      3
1896.00 AUT   Austria                Fencing     1       0       2      3
1896.00 AUT   Austria    Swimming (Aquatics)     1       1       0      2
1896.00 DEN   Denmark              Athletics     1       2       3      6
1896.00 FRA    France              Athletics     0       1       1      2
1896.00 FRA    France Cycling Road (Cycling)     4       1       1      6
1896.00 FRA    France                Fencing     1       2       0      3
1896.00 GBR        UK              Athletics     1       2       2      5
1896.00 GBR        UK Cycling Road (Cycling)     0       1       1      2
1896.00 GBR        UK                 Tennis     2       0       0      2


## 5. Enrichissement : population + continent

On ajoute deux features structurelles pour le modele ML :
- **Population** (en millions) du pays a l'annee donnee, via Banque Mondiale
- **Continent** (mapping manuel pour les 158 pays medailles)

In [6]:
# Transformation populations (wide -> long)
year_cols = [c for c in populations.columns if c.isdigit()]
pop_long = populations.melt(
    id_vars=["Country Name", "Country Code"],
    value_vars=year_cols, var_name="Year", value_name="Population"
)
pop_long["Year"] = pop_long["Year"].astype(int)
pop_long["Population_million"] = pop_long["Population"] / 1_000_000

# Mapping NOC -> Country Code ISO (pour matcher avec World Bank)
noc_to_iso = {
    # Correspondances directes
    "AFG":"AFG","ALB":"ALB","ARG":"ARG","ARM":"ARM","AUS":"AUS","AUT":"AUT",
    "AZE":"AZE","BEL":"BEL","BIH":"BIH","BLR":"BLR","BOL":"BOL","BRA":"BRA",
    "CAN":"CAN","CHN":"CHN","COL":"COL","CRI":"CRI","CUB":"CUB","CYP":"CYP",
    "CZE":"CZE","ECU":"ECU","EGY":"EGY","ESP":"ESP","EST":"EST","ETH":"ETH",
    "FIN":"FIN","FRA":"FRA","GBR":"GBR","GEO":"GEO","GHA":"GHA","HUN":"HUN",
    "IND":"IND","IRL":"IRL","IRN":"IRN","ISL":"ISL","ISR":"ISR","ITA":"ITA",
    "JAM":"JAM","JPN":"JPN","KAZ":"KAZ","KEN":"KEN","KGZ":"KGZ","LTU":"LTU",
    "LVA":"LVA","MAR":"MAR","MEX":"MEX","MNE":"MNE","NOR":"NOR","NZL":"NZL",
    "PAK":"PAK","PER":"PER","POL":"POL","PRT":"PRT","ROU":"ROU","RUS":"RUS",
    "SRB":"SRB","SVK":"SVK","SVN":"SVN","SWE":"SWE","TUN":"TUN","TUR":"TUR",
    "UKR":"UKR","UZB":"UZB","VEN":"VEN","ZAF":"ZAF","ZWE":"ZWE",
    # Mappings differents
    "ALG":"DZA","BAH":"BHS","BAR":"BRB","BUL":"BGR","CHI":"CHL","CRO":"HRV",
    "DEN":"DNK","DOM":"DOM","ESA":"SLV","FIJ":"FJI","GER":"DEU","GRE":"GRC",
    "GUA":"GTM","HAI":"HTI","INA":"IDN","KOR":"KOR","KSA":"SAU","LAT":"LVA",
    "MGL":"MNG","NED":"NLD","NGR":"NGA","PHI":"PHL","POR":"PRT","PUR":"PRI",
    "RSA":"ZAF","SIN":"SGP","SLO":"SVN","SUI":"CHE","TPE":"TWN","TRI":"TTO",
    "UAE":"ARE","USA":"USA","URU":"URY","VIE":"VNM",
}

agg["ISO_Code"] = agg["NOC"].map(noc_to_iso)
agg = agg.merge(
    pop_long[["Country Code", "Year", "Population_million"]],
    left_on=["ISO_Code", "Year"], right_on=["Country Code", "Year"],
    how="left"
).drop(columns=["Country Code", "ISO_Code"])

print(f"Population trouvee : {agg['Population_million'].notna().sum():,} / {len(agg):,}")

Population trouvee : 4,201 / 6,871


In [7]:
# Mapping continent (groupe par region)
continent_map = {
    "Europe": ["FRA","GBR","GER","ITA","ESP","NED","BEL","SUI","AUT","SWE","NOR","DEN","FIN",
                "POL","CZE","SVK","SVN","CRO","SRB","BUL","ROU","HUN","GRE","POR","IRL","ISL",
                "EST","LAT","LTU","BLR","UKR","RUS","RSA","ALB","BIH","MKD","MNE","TUR","GEO",
                "AZE","ARM","MDA","CYP","MLT","LUX","MON","SMR","AND","LIE",
                "URS","FRG","GDR","TCH","YUG","EUN","ROC","SCG","SCG","OAR","BOH"],
    "Asia": ["CHN","JPN","KOR","PRK","IND","PAK","BAN","SRI","IRI","IRQ","KAZ","UZB","KGZ",
              "TJK","TKM","MGL","TPE","HKG","SIN","MAS","INA","PHI","VIE","THA","CAM","LAO",
              "MYA","NEP","BHU","MDV","KUW","KSA","UAE","QAT","BAH","OMA","YEM","LBN","JOR",
              "SYR","ISR","PSE","AFG","TLS"],
    "Africa": ["EGY","MAR","ALG","TUN","LBA","SUD","SSD","ERI","ETH","KEN","UGA","TAN","RWA",
                "BDI","COD","CGO","GAB","CMR","CAF","TCD","NGR","NIG","BEN","TOG","GHA","CIV",
                "BUR","LBR","SLE","GUI","SEN","GAM","CPV","MTN","MLI","MAD","MAW","MOZ","ZAM",
                "ZIM","BOT","NAM","LES","SWZ","ANG","COM","DJI","SEY","STP","GEQ","RHO"],
    "North America": ["USA","CAN","MEX","CUB","JAM","HAI","DOM","PUR","BAH","TRI","BAR","ANT",
                       "GRN","DMA","VIN","SKN","GUA","ESA","HON","NCA","CRC","PAN","BIZ","BER",
                       "CAY","ARU","ISV","AHO","IVB","TKS"],
    "South America": ["ARG","BRA","CHI","COL","ECU","PER","URU","VEN","PAR","BOL","GUY","SUR"],
    "Oceania": ["AUS","NZL","FIJ","SAM","TGA","KIR","TUV","NRU","PLW","SOL","VAN","PNG","ASA",
                 "MHL","FSM","COK","GUM","ANZ"],
}
noc_to_continent = {noc: cont for cont, nocs in continent_map.items() for noc in nocs}

agg["Continent"] = agg["NOC"].map(noc_to_continent).fillna("Unknown")
print("Repartition par continent :")
print(agg["Continent"].value_counts().to_string())

Repartition par continent :
Continent
Europe           4349
North America     868
Asia              860
Oceania           313
South America     250
Africa            167
Unknown            64


## 6. Sauvegarde du dataset final

`medals_per_country_sport.csv` est notre **unique fichier** pour le ML et la viz.

In [8]:
# Reorganiser et sauvegarder
agg_final = agg[["Year", "NOC", "Country", "Continent", "Sport",
                   "Gold", "Silver", "Bronze", "Total", "Population_million"]]

output_path = os.path.join(PROCESSED_DIR, "medals_per_country_sport.csv")
agg_final.to_csv(output_path, index=False)

size_kb = os.path.getsize(output_path) / 1024
print(f"medals_per_country_sport.csv sauvegarde : {agg_final.shape[0]:,} lignes, {size_kb:.0f} Ko")
print(f"\nColonnes finales : {list(agg_final.columns)}")
print(f"\nApercu :")
print(agg_final.head(10).to_string(index=False))

medals_per_country_sport.csv sauvegarde : 6,871 lignes, 371 Ko

Colonnes finales : ['Year', 'NOC', 'Country', 'Continent', 'Sport', 'Gold', 'Silver', 'Bronze', 'Total', 'Population_million']

Apercu :
   Year NOC   Country Continent                  Sport  Gold  Silver  Bronze  Total  Population_million
1896.00 AUS Australia   Oceania              Athletics     2       0       1      3                 NaN
1896.00 AUT   Austria    Europe                Fencing     1       0       2      3                 NaN
1896.00 AUT   Austria    Europe    Swimming (Aquatics)     1       1       0      2                 NaN
1896.00 DEN   Denmark    Europe              Athletics     1       2       3      6                 NaN
1896.00 FRA    France    Europe              Athletics     0       1       1      2                 NaN
1896.00 FRA    France    Europe Cycling Road (Cycling)     4       1       1      6                 NaN
1896.00 FRA    France    Europe                Fencing     1       2   

In [9]:
# Verifications de qualite finales
print("=== CONTROLES QUALITE ===")
print(f"\nDoublons (NOC, Year, Sport) : {agg_final.duplicated(subset=['NOC','Year','Sport']).sum()}")
print(f"NaN par colonne :")
print(agg_final.isna().sum().to_string())
print(f"\nPlages :")
print(f"  Annees : {agg_final['Year'].min()} - {agg_final['Year'].max()}")
print(f"  Pays   : {agg_final['NOC'].nunique()}")
print(f"  Sports : {agg_final['Sport'].nunique()}")
print(f"\nTotal medailles distribuees : {agg_final['Total'].sum():,}")

=== CONTROLES QUALITE ===

Doublons (NOC, Year, Sport) : 0
NaN par colonne :
Year                     0
NOC                      0
Country                 33
Continent                0
Sport                    0
Gold                     0
Silver                   0
Bronze                   0
Total                    0
Population_million    2670

Plages :
  Annees : 1896.0 - 2024.0
  Pays   : 157
  Sports : 90

Total medailles distribuees : 17,802


## Conclusion

Le dataset final est **propre, agrege, et leger** :
- 1 seul fichier `medals_per_country_sport.csv` (~1 Mo)
- 1 ligne = (NOC, Year, Sport) avec compteurs Or/Argent/Bronze/Total
- Enrichi avec population et continent
- 0 doublon, 0 NaN sur les champs critiques

C'est ce dataset qui alimente le modele ML du notebook suivant et le site web.